In [1]:
import os
os.environ["HF_HOME"] = "/home/yandex/APDL2425a/group_12/gorodissky/.cache/huggingface"
print(f"HF_HOME set to:\t\t {os.environ['HF_HOME']}")

import torch
print(f"CUDA available: \t{torch.cuda.is_available()}")
print(f"Torch version: \t\t{torch.__version__}")
if torch.cuda.is_available():
    print(f"Number of CUDA devices\t {torch.cuda.device_count()}")
    print(f"CUDA device:\t\t {torch.cuda.get_device_name(torch.cuda.current_device())}")

HF_HOME set to:		 /home/yandex/APDL2425a/group_12/gorodissky/.cache/huggingface
CUDA available: 	True
Torch version: 		2.7.1+cu126
Number of CUDA devices	 4
CUDA device:		 NVIDIA GeForce GTX TITAN X


In [2]:
from sentence_transformers import SentenceTransformer
from datasets import load_dataset, load_from_disk, disable_progress_bar
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os
from transformers import AutoModelForCausalLM, AutoModel, AutoTokenizer, AutoConfig, DataCollatorWithPadding, Qwen2_5_VLConfig
from probes import eval_baseline, eval, create_dataset_baseline, create_datasets
from collections import defaultdict
import numpy as np

In [3]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
dataset_name = "allenai/WildChat-1M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, dtype="auto", device_map="auto")
df = load_from_disk(f"data/preprocessed/{dataset_name}/{model_name}_L=256").to_pandas()
print(f"Loaded {model_name} on {model.device} with precision {model.dtype}")
print(f"Loaded preprocessed dataset {dataset_name} with {len(df)} samples")

Loaded Qwen/Qwen2.5-0.5B-Instruct on cuda:1 with precision torch.bfloat16
Loaded preprocessed dataset allenai/WildChat-1M with 606968 samples


In [6]:
X, Y = create_datasets(
    model=model,
    model_name=model_name,
    tokenizer=tokenizer,
    dataset_name=dataset_name,
    dataset_size=64
)

Processed batch 1/2
Processed batch 2/2


In [8]:
print(X.keys())
print(X['last'].keys())
print(Y.keys())

dict_keys(['last'])
dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23])
dict_keys(['response_token_ids', 'lengths'])


In [ ]:
assert model.config.num_hidden_layers == len(X['last']), "Number of layers mismatch"
for acts in X["last"].values():
    assert len(acts.shape) == 2, "Hidden states tesnor have other than 2 dimensions"
    assert acts.shape[0] == 64, "Number of hidden states is not equal dataset size"
    print(acts.shape[1])
    assert acts.shape[1] == model.config.hidden_size, "Dimension of hidden states don't model hidden dim"

896


AttributeError: 'Qwen2Config' object has no attribute 'hidden_dim'

In [12]:
for q, a in zip(df["input_ids"][:5],Y["response_token_ids"][:5]):
    print("Q:", tokenizer.decode(q, skip_special_tokens=False))
    print("A:", tokenizer.decode(a, skip_special_tokens=False))
    print("*"* 100)

Q: <|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Crea una imagen de una mujer corriente por la playa en bikini <|im_end|>
<|im_start|>assistant

A: When responding to a review from a security company about their excellent work, it's important to acknowledge the positive aspects while also highlighting areas where improvements could be made. Here’s a constructive way to respond:

---

I appreciate your feedback on our services. Your reviews have been very encouraging and I am grateful for the insights they provide into what we can improve upon.

Your team has consistently demonstrated exceptional professionalism and attention to detail in all of our projects. The quality of our data protection measures is commendable, and you've shown a keen eye for potential vulnerabilities that were not immediately apparent.

However, there are always areas for improvement. For example, we should consider implementing more robust enc

In [14]:
model.config

Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "max_window_layers": 21,
  "model_type": "qwen2",
  "num_attention_heads": 14,
  "num_hidden_layers": 24,
  "num_key_value_heads": 2,
  "rms_